# Neural Network

## Step 0: Library & Data Imports
For the neural network, only two libraries were used: pandas and numpy. They are useful for a number of things, from loading the data from the csv file to matrix multiplication.

No other libraries were used in order to showcase a "from scratch" implementation that demonstrates course material comprehension.

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")

## Step 1: Pre-Treatment

The dataset has 14 binary features, 3 quantitative features and 4 ordinal variables. While keeping the ordinality makes sense for GenHlth, the patient's rating of their general health, and Education, age and income can quite naturally be transformed back to quantitative data. The only issue was choosing a suitable number for the last, open-ended category of each. For age, 85 was chosen as an estimation of what the average age of that group might be. For income, the number was brought up a bit from the "natural progression" in order to better reflect the open-endedness.

Afterward, the dataframe is shuffled and split into a training and validation set.

Finally, the features are standardized by changing values to their z score.


In [13]:
age_map = {1: 21, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52, 8: 57, 9: 62, 10: 67, 11: 72, 12: 77, 13: 85}
income_map = {1: 5000, 2: 12500, 3: 17500, 4: 22500, 5: 30000, 6: 42500, 7: 62500, 8: 87500}
df["IncomeMid"] = df["Income"].map(income_map)
df["AgeMid"] = df["Age"].map(age_map)

# Beginning by splitting the dataframe into two parts, using proportions 8:2 for training:validation

np.random.seed(42)
shuffled_df = df.sample(frac=1).reset_index(drop=True)
shuffled_df = shuffled_df.drop(columns=["Income","Age"])

train_df = shuffled_df.iloc[:(int(len(shuffled_df)*(8/10)))].copy()
val_df = shuffled_df.iloc[int(len(shuffled_df)*(8/10)):].copy()

# Standardizing
for category in ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Education", "IncomeMid", "AgeMid"]:
    mean_tr = train_df[category].mean()
    std_tr = train_df[category].std()
    train_df[category] = (train_df[category] - mean_tr) / std_tr
    val_df[category] = (val_df[category] - mean_tr) / std_tr

# Separating Xs and Ys
X_train = train_df.drop(columns=["Diabetes_binary"])
Y_train = train_df["Diabetes_binary"]
Y_train = np.asarray(Y_train).reshape(-1,1)
X_val = val_df.drop(columns=["Diabetes_binary"])
Y_val = val_df["Diabetes_binary"]
Y_val = np.asarray(Y_val).reshape(-1,1)

## Activation functions:

The sigmoid and reLu functions as seen in the course are implemented here, along with their derivatives.
<img src="../data/sigmoidVreLu.png">
_Description_: Sigmoid function on the left and reLu function on the right.
_Source_: Artificial Intelligence: A Modern Approach, 4th Edition, 2021. Peter Norvig and Stuart J. Russell. Pearson Education. p. 803

In [14]:
def sigmoid(x):
    x = np.clip(x,-500,500)     #prevent overflow
    return 1 / (1 + np.exp(-x))
def reLu(x):
    return np.maximum(0, x)
def reLu_deriv(x):
    return (x>0).astype(float)
def sigmoid_deriv(x):
    return x * (1-x)

Following a very simple intuition of neural networks

At forward pass, we multiply

In [15]:
learning_rate=0.02
epochs = 1000

# numpy is faster for matrix multiplication, which we'll be doing tons of, so converting df into np arrays
X_train_np = X_train.to_numpy()
Y_train_np = np.asarray(Y_train).reshape(-1,1)
X_val_np = X_val.to_numpy()
Y_val_np = np.asarray(Y_val).reshape(-1,1)

def addIntercept(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

X_train_np = addIntercept(X_train_np)
X_val_np = addIntercept(X_val_np)

def binaryCrossEntropy(y_true, y_pred):
    epsilon = 1e-12
    y_pred = np.clip(y_pred, epsilon, 1-epsilon)
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def makeBatches(X, Y, batchSize):
    m = X.shape[0]
    batches = []
    for start in range(0, m, batchSize):
        end = start + batchSize
        X_batch = X[start:end]
        Y_batch = Y[start:end]
        batches.append((X_batch, Y_batch))
    return batches

def forwardPropagation(X_batch, W1, W2):
    Z1 = X_batch @ W1           # matrix multiply X and W1
    A1 = reLu(Z1)               # apply reLu function
    A1_i = addIntercept(A1)     # add intercept
    Z2 = A1_i @ W2              # matrix multiply result and W2
    A2 = sigmoid(Z2)            # apply sigmoid function
    cache = {"X_batch": X_batch, "Z1": Z1, "A1": A1, "A1_i":A1_i, "Z2": Z2, "A2": A2}
    return A2, cache

def backwardPropagation(Y_batch, W2, cache):
    # unpack cache
    X_batch = cache["X_batch"]
    Z1 = cache["Z1"]
    A1 = cache["A1"]
    A1_i = cache["A1_i"]
    A2 = cache["A2"]
    m = X_batch.shape[0]

    #output error
    dZ2 = A2 - Y_batch
    # gradient for w2
    dW2 = (A1_i.T @ dZ2) / m
    # error passed back to hidden layer
    dA1 = dZ2 @ W2[1:,:].T
    # hidden layer error
    dZ1 = dA1 * reLu_deriv(Z1)
    # gradient for W1
    dW1 = (X_batch.T @ dZ1) / m

    return dW1, dW2

def predict(X, W1, W2):
    A2, cache = forwardPropagation(X, W1, W2)
    y_class = (A2 > 0.5).astype(int)
    return y_class, A2

best_val_loss = float("inf")
best_W1 = None
best_W2 = None
patience_counter = 0
best_epoch = 0
patience = 30
min_delta = 0.0001


np.random.seed(137)
input_size = X_train_np.shape[1]
output_size = 1
hidden_layer_size = 30
# W1 is the connection between input to hidden. shape (22,15)
W1 = np.random.randn(input_size, hidden_layer_size) * np.sqrt(2 / input_size)
# W2 is the connection between hidden to output. shape (15,1)
W2 = np.random.randn(hidden_layer_size+1, output_size) * np.sqrt(2 / hidden_layer_size)

n_samples = X_train_np.shape[0]
batchSize = 128
#print("learning rate: ", learning_rate)
for epoch in range(epochs):
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    X_shuffled = X_train_np[indices]
    Y_shuffled = Y_train_np[indices]
    for start in range(0, n_samples, batchSize):
        end = start + batchSize
        X_batch = X_shuffled[start:end]
        Y_batch = Y_shuffled[start:end]
        m = X_batch.shape[0]

        A2, cache = forwardPropagation(X_batch, W1, W2)
        dW1, dW2 = backwardPropagation(Y_batch, W2, cache)

        # update weights
        W2 = W2 - learning_rate * dW2
        W1 = W1 - learning_rate * dW1

    # check current loss & accuracy on validation set
    val_class, val_pred = predict(X_val_np, W1, W2)
    val_loss = binaryCrossEntropy(Y_val_np, val_pred)
    val_acc = accuracy(Y_val_np, val_class)

    # if
    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        best_val_acc = val_acc
        best_W1 = W1.copy()
        best_W2 = W2.copy()
        patience_counter = 0
        best_epoch = epoch
    else:
        patience_counter += 1

    # print to check progress
    if epoch % 50 == 49:
        train_class, train_pred = predict(X_train_np, W1, W2)
        train_loss = binaryCrossEntropy(Y_train_np, train_pred)
        train_acc = accuracy(Y_train_np, train_class)
        Z1_val = X_val_np @ W1
        A1_val = reLu(Z1_val)
        A1_val_i = addIntercept(A1_val)
        Z2_val = A1_val_i @ W2
        val_pred = sigmoid(Z2_val)
        val_class = (val_pred > 0.5).astype(int)

        val_loss = binaryCrossEntropy(Y_val_np, val_pred)
        val_acc = accuracy(Y_val_np, val_class)

        print(
            "Epoch:", epoch,
            "| Train loss:", round(train_loss, 4),
            "| Train acc:", round(train_acc, 4),
            "| Val loss:", round(val_loss, 4),
            "| Val acc:", round(val_acc, 4)
        )
    if patience_counter >= patience:
        print("Early stopping at epoch:", epoch)
        print("Best epoch:", best_epoch)
        print("Best validation loss:", best_val_loss)
        print("Best validation accuracy:", best_val_acc)
        break




Epoch: 49 | Train loss: 0.5014 | Train acc: 0.753 | Val loss: 0.5079 | Val acc: 0.7492
Epoch: 99 | Train loss: 0.4993 | Train acc: 0.7541 | Val loss: 0.506 | Val acc: 0.7493
Epoch: 149 | Train loss: 0.4985 | Train acc: 0.7548 | Val loss: 0.5054 | Val acc: 0.7505
Early stopping at epoch: 183
Best epoch: 153
Best validation loss: 0.505167924784269
Best validation accuracy: 0.7507603083669283


In [16]:
y_class, blah = predict(X_val_np, best_W1, best_W2)
y_true = np.asarray(Y_val_np).reshape(-1)
y_class = np.asarray(y_class).reshape(-1)
TP = np.sum((y_true == 1) & (y_class == 1))
FP = np.sum((y_true == 0) & (y_class == 1))
TN = np.sum((y_true == 0) & (y_class == 0))
FN = np.sum((y_true == 1) & (y_class == 0))
print("True positives:", TP)
print("False positives:", FP)
print("True negatives:", TN)
print("False negatives:", FN)

True positives: 5513
False positives: 2059
True negatives: 5102
False negatives: 1465
